In [0]:
# importar librerias
from pyspark.sql import functions as F
from delta.tables import DeltaTable

FACT_TRANSACTIONS = "banco.banking_analytics.fact_transactions"

In [0]:
# Analizar el plan de ejecución
df_perf_baseline = (
    spark.table(FACT_TRANSACTIONS)
    .filter(F.col("status") == "COMPLETED")
    .groupBy("currency")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount")
    )
)

df_perf_baseline.explain("formatted")

In [0]:
display(df_perf_baseline)

In [0]:
# Medir el tiempo de ejecución
import time

start_time = time.perf_counter()

results = df_perf_baseline.collect()

end_time = time.perf_counter()

execution_time = end_time - start_time

print(f"Tiempo de ejecución: {execution_time:.4f} segundos")

In [0]:
# Analizar estadisticas de ejecución
df_perf_baseline.explain("cost")

In [0]:
# Verificar estadisticas de la tabla
spark.sql("""
    DESCRIBE DETAIL banco.banking_analytics.fact_transactions
""").display()

In [0]:
%sql
USE CATALOG banco;
SHOW TABLE EXTENDED IN banco.banking_analytics
LIKE 'fact_transactions';


In [0]:
df_file_sizes = spark.sql(f"""
    DESCRIBE DETAIL {FACT_TRANSACTIONS}
""").select(
    "numFiles",
    "sizeInBytes",
    "partitionColumns",
    "clusteringColumns"
)

display(df_file_sizes)

In [0]:
display(
    spark.table(FACT_TRANSACTIONS)
    .select("transaction_id")
    .limit(10)
)

In [0]:
# Simular Small File Problems
df_small_files = (
    spark.table(FACT_TRANSACTIONS)
    .repartition(20)
)

print(f"Particiones del DataFrame: {df_small_files.select(F.spark_partition_id()).distinct().count()}")
print(f"Registros: {df_small_files.count()}")

In [0]:
# Generar archivos pequeños
SMALL_FILES_TABLE = "banco.banking_analytics.fact_transactions_small_files"

(
    df_small_files.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SMALL_FILES_TABLE)
)

print("Tabla experimental creada correctamente.")

In [0]:
# Inspeccionar los pequeños archivos generados
df_small_detail = spark.sql(f"""
    DESCRIBE DETAIL {SMALL_FILES_TABLE}
""")

display(
    df_small_detail.select(
        "numFiles",
        "sizeInBytes",
        "partitionColumns"
    )
)

In [0]:
# Aplicando OPTIMIZE
spark.sql(f"""
    OPTIMIZE {SMALL_FILES_TABLE}
""").display()

In [0]:
display(
    spark.sql(f"DESCRIBE DETAIL {SMALL_FILES_TABLE}")
    .select("numFiles", "sizeInBytes")
)

In [0]:
display(
    DeltaTable.forName(spark, SMALL_FILES_TABLE)
    .history(5)
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
    .orderBy(F.col("version").desc())
)

In [0]:
df_after_optimize = spark.table(SMALL_FILES_TABLE)

print(f"Total de registros: {df_after_optimize.count()}")

print(
    "Transacciones únicas:",
    df_after_optimize
    .select("transaction_id")
    .distinct()
    .count()
)

In [0]:
SMALL_FILES_TABLE = "banco.banking_analytics.fact_transactions_small_files"

spark.sql(f"""
DESCRIBE DETAIL {SMALL_FILES_TABLE}
""").display()

In [0]:
spark.sql(f"""
OPTIMIZE {SMALL_FILES_TABLE}
ZORDER BY (customer_key, transaction_date)
""").display()

In [0]:
# Comparar Z-Order con Liquid Clustering
ZORDER_TABLE = "banco.banking_analytics.fact_transactions_zorder"

LIQUID_TABLE = "banco.banking_analytics.fact_transactions_liquid"

In [0]:
# Tabla para aplicar Z-order
(
    spark.table(FACT_TRANSACTIONS)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(ZORDER_TABLE)
)

In [0]:
# Tabla para aplicar liquid clustering
spark.sql(f"""
CREATE OR REPLACE TABLE {LIQUID_TABLE}
CLUSTER BY (customer_key, transaction_date)
AS
SELECT *
FROM {FACT_TRANSACTIONS}
""")

In [0]:
# Validar integridad de datos
tables_to_validate = [
    FACT_TRANSACTIONS,
    ZORDER_TABLE,
    LIQUID_TABLE
]

for table_name in tables_to_validate:
    df = spark.table(table_name)

    total_rows = df.count()
    unique_transactions = (
        df.select("transaction_id")
        .distinct()
        .count()
    )

    print(f"\nTabla: {table_name}")
    print(f"Total registros: {total_rows}")
    print(f"Transacciones únicas: {unique_transactions}")

In [0]:
# Inspeccionar ambas tablas
for table_name in [ZORDER_TABLE, LIQUID_TABLE]:

    print(f"\n========== {table_name} ==========")

    spark.sql(f"""
        DESCRIBE DETAIL {table_name}
    """).select(
        "numFiles",
        "sizeInBytes",
        "partitionColumns",
        "clusteringColumns",
        "clusterByAuto"
    ).display()

In [0]:
# Aplicando Zorder
spark.sql(f"""
OPTIMIZE {ZORDER_TABLE}
ZORDER BY (customer_key, transaction_date)
""").display()

In [0]:
# Ejecutar la tabla liquid
spark.sql(f"""
OPTIMIZE {LIQUID_TABLE}
""").display()

In [0]:
# Historial Delta
for table_name in [ZORDER_TABLE, LIQUID_TABLE]:

    print(f"\n========== {table_name} ==========")

    spark.sql(f"""
        DESCRIBE HISTORY {table_name}
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    ).display()

In [0]:
for table_name in [ZORDER_TABLE, LIQUID_TABLE]:

    print(f"\n========== {table_name} ==========")

    spark.sql(f"""
        DESCRIBE DETAIL {table_name}
    """).select(
        "numFiles",
        "sizeInBytes",
        "partitionColumns",
        "clusteringColumns"
    ).display()

In [0]:
BENCHMARK_BASE = (
    "banco.banking_analytics.fact_transactions_benchmark_base"
)

In [0]:
df_benchmark = (
    spark.table(FACT_TRANSACTIONS)
    .crossJoin(
        spark.range(50).withColumnRenamed("id", "copy_id")
    )
    .withColumn(
        "transaction_id",
        F.concat(
            F.col("transaction_id"),
            F.lit("_"),
            F.col("copy_id").cast("string")
        )
    )
    .drop("copy_id")
    .repartition(4)
)

In [0]:
print("Registros:", df_benchmark.count())
print("Particiones:", df_benchmark.select(F.spark_partition_id()).distinct().count())

In [0]:
# Guardarlo como tabla delta
(
    df_benchmark.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BENCHMARK_BASE)
)

In [0]:
# Validamos
spark.sql(f"""
DESCRIBE DETAIL {BENCHMARK_BASE}
""").select(
    "numFiles",
    "sizeInBytes",
    "partitionColumns",
    "clusteringColumns"
).display()

In [0]:
BENCHMARK_ZORDER = (
    "banco.banking_analytics.fact_transactions_benchmark_zorder"
)

BENCHMARK_LIQUID = (
    "banco.banking_analytics.fact_transactions_benchmark_liquid"
)

In [0]:
(
    spark.table(BENCHMARK_BASE)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BENCHMARK_ZORDER)
)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {BENCHMARK_LIQUID}
CLUSTER BY (customer_key, transaction_date)
AS
SELECT *
FROM {BENCHMARK_BASE}
""")

In [0]:
for table_name in [
    BENCHMARK_BASE,
    BENCHMARK_ZORDER,
    BENCHMARK_LIQUID
]:
    df = spark.table(table_name)

    print(f"\nTabla: {table_name}")
    print("Registros:", df.count())
    print(
        "Transacciones únicas:",
        df.select("transaction_id").distinct().count()
    )

In [0]:
for table_name in [BENCHMARK_ZORDER, BENCHMARK_LIQUID]:

    print(f"\nTabla: {table_name}")

    spark.sql(f"""
        DESCRIBE DETAIL {table_name}
    """).select(
        "numFiles",
        "sizeInBytes",
        "partitionColumns",
        "clusteringColumns"
    ).display()

In [0]:
spark.sql(f"""
OPTIMIZE {BENCHMARK_ZORDER}
ZORDER BY (customer_key, transaction_date)
""").display()

In [0]:
spark.sql(f"""
OPTIMIZE {BENCHMARK_LIQUID}
""").display()


In [0]:
BENCHMARK_DISTRIBUTED = (
    "banco.banking_analytics.fact_transactions_benchmark_distributed"
)

df_benchmark_distributed = (
    spark.table(BENCHMARK_BASE)
    .repartition(32, "customer_key")
)

In [0]:
print(
    "Particiones Spark:",
    df_benchmark_distributed.select(F.spark_partition_id()).distinct().count()
)

In [0]:
(
    df_benchmark_distributed
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BENCHMARK_DISTRIBUTED)
)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE DETAIL {BENCHMARK_DISTRIBUTED}
    """)
)

In [0]:
# Probar ZOrder
spark.sql(f"""
    OPTIMIZE {BENCHMARK_DISTRIBUTED}
    ZORDER BY (customer_key, transaction_date)
""")

In [0]:
display(
    spark.sql(f"""
        DESCRIBE HISTORY {BENCHMARK_DISTRIBUTED}
    """)
)

In [0]:
df_zorder_validation = spark.table(BENCHMARK_DISTRIBUTED)

print("Total registros:", df_zorder_validation.count())

print(
    "IDs únicos:",
    df_zorder_validation.select("transaction_id").distinct().count()
)

In [0]:
# Ahora probar con Liquid Cluster
spark.sql(f"""
    CREATE OR REPLACE TABLE {BENCHMARK_LIQUID}
    CLUSTER BY (customer_key, transaction_date)
    AS
    SELECT *
    FROM {BENCHMARK_BASE}
""")

In [0]:
display(
    spark.sql(f"""
        DESCRIBE DETAIL {BENCHMARK_LIQUID}
    """)
)

In [0]:
spark.sql(f"""
    OPTIMIZE {BENCHMARK_LIQUID}
""")

In [0]:
display(
    spark.sql(f"""
        DESCRIBE HISTORY {BENCHMARK_LIQUID}
    """)
)

In [0]:
df_liquid_validation = spark.table(BENCHMARK_LIQUID)

print("Total registros:", df_liquid_validation.count())

print(
    "IDs únicos:",
    df_liquid_validation
    .select("transaction_id")
    .distinct()
    .count()
)

In [0]:
# Analizando Shuffle
from pyspark.sql import functions as F

df_shuffle_source = spark.table(BENCHMARK_BASE)

df_shuffle_baseline = (
    df_shuffle_source
    .groupBy("currency")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount")
    )
)

print(
    "Particiones de entrada:",
    df_shuffle_source.select("transaction_id").distinct().count()
)

df_shuffle_baseline.explain("formatted")

In [0]:
print(
    "Archivos físicos de la tabla:",
    spark.sql(
        f"DESCRIBE DETAIL {BENCHMARK_BASE}"
    ).select("numFiles").first()[0]
)

print(
    "Particiones de entrada:",
    df_shuffle_source.select(F.spark_partition_id()).distinct().count()
)

print(
    "Particiones de Shuffle configuradas:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

In [0]:
# Comparar plan inicial con plan final
result_shuffle = df_shuffle_baseline.collect()

print("Resultado de la agregación:")
for row in result_shuffle:
    print(row)

In [0]:
df_shuffle_baseline.explain("formatted")

In [0]:
configs = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize",
    "spark.sql.shuffle.partitions"
]

for config in configs:
    try:
        print(f"{config}: {spark.conf.get(config)}")
    except Exception as e:
        print(f"{config}: No disponible ({str(e)[:120]})")

In [0]:
df_partition_source = spark.table(BENCHMARK_BASE)

df_repartitioned = df_partition_source.repartition(8)

df_coalesced = df_partition_source.coalesce(2)

print("Particiones originales:",
      df_partition_source.select(F.spark_partition_id()).distinct().count())

print("Particiones después de repartition:",
      df_repartitioned.select(F.spark_partition_id()).distinct().count())

print("Particiones después de coalesce:",
      df_coalesced.select(F.spark_partition_id()).distinct().count())

In [0]:
print("===== REPARTITION =====")
df_repartitioned.explain("formatted")

print("===== COALESCE =====")
df_coalesced.explain("formatted")

In [0]:
# Repartition mediante Hash
df_repartition_customer = (
    df_partition_source.repartition(8, "customer_key")
)

print(
    "Particiones:",
    df_repartition_customer.select(F.spark_partition_id()).distinct().count()
)

df_repartition_customer.explain("formatted")

In [0]:
# Analizando el broadcast join
# Empezamos recogiendo un join regular
from pyspark.sql import functions as F

df_fact_join = spark.table(BENCHMARK_BASE).alias("f")

df_customer_join = (
    spark.table("banco.banking_analytics.dim_customer")
    .alias("c")
)

df_join_normal = (
    df_fact_join
    .join(
        df_customer_join,
        F.col("f.customer_key") == F.col("c.customer_key"),
        "inner"
    )
    .select(
        F.col("f.transaction_id"),
        F.col("f.customer_key"),
        F.col("f.amount"),
        F.col("c.first_name"),
        F.col("c.last_name")
    )
)

df_join_normal.explain("formatted")

In [0]:
%sql
SELECT * FROM banco.banking_analytics.dim_customer
LIMIT 10;

In [0]:
df_join_normal = (
    df_fact_join
    .join(
        df_customer_join,
        F.col("f.customer_key") == F.col("c.customer_key"),
        "inner"
    )
    .select(
        F.col("f.transaction_id"),
        F.col("f.customer_key"),
        F.col("f.amount"),
        F.col("c.customer_name")
    )
)

df_join_normal.explain("formatted")

In [0]:
# Forzar el uso de Broadcast join
from pyspark.sql.functions import broadcast

df_join_broadcast = (
    df_fact_join
    .join(
        broadcast(df_customer_join),
        F.col("f.customer_key") == F.col("c.customer_key"),
        "inner"
    )
    .select(
        F.col("f.transaction_id"),
        F.col("f.customer_key"),
        F.col("f.amount"),
        F.col("c.customer_name")
    )
)

df_join_broadcast.explain("formatted")

In [0]:
# Validar ambos resultados
print("JOIN automático:", df_join_normal.count())
print("JOIN Broadcast:", df_join_broadcast.count())

In [0]:
# Analizando Data Skew (Sesgo de datos)
from pyspark.sql import functions as F

# Crear un conjunto de datos con distribución desbalanceada
df_skew = (
    spark.range(1_000_000)
    .withColumn(
        "customer_key",
        F.when(F.col("id") < 700_000, 1)
         .otherwise((F.col("id") % 1000) + 2)
    )
    .repartition(8, "customer_key")
)

print("Total registros:", df_skew.count())

print("Particiones:", df_skew.select(F.spark_partition_id()).distinct().count())

In [0]:
df_partition_distribution = (
    df_skew
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
    .groupBy("partition_id")
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("customer_key").alias("distinct_customers")
    )
    .orderBy("partition_id")
)

display(df_partition_distribution)

In [0]:
df_skew_salted = (
    df_skew
    .withColumn(
        "salt",
        F.when(
            F.col("customer_key") == 1,
            F.pmod(F.col("id"), F.lit(8))
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "customer_key_salted",
        F.concat_ws(
            "_",
            F.col("customer_key").cast("string"),
            F.col("salt").cast("string")
        )
    )
    .repartition(8, "customer_key_salted")
)

print("Total registros:", df_skew_salted.count())
print("Particiones:", df_skew_salted.select(F.spark_partition_id()).distinct().count())

In [0]:
# Medir la nueva distribucion
df_salted_distribution = (
    df_skew_salted
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
    .groupBy("partition_id")
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("customer_key").alias("distinct_customers"),
        F.countDistinct("customer_key_salted").alias("distinct_salted_keys")
    )
    .orderBy("partition_id")
)

display(df_salted_distribution)

In [0]:
df_hot_customer_distribution = (
    df_skew_salted
    .filter(F.col("customer_key") == 1)
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
    .groupBy("partition_id", "salt")
    .agg(
        F.count("*").alias("total_records")
    )
    .orderBy("partition_id", "salt")
)

display(df_hot_customer_distribution)

In [0]:
df_skew_direct = (
    df_skew
    .repartition(8, "id")
)

df_direct_distribution = (
    df_skew_direct
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
    .groupBy("partition_id")
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("customer_key").alias("distinct_customers")
    )
    .orderBy("partition_id")
)

display(df_direct_distribution)

In [0]:

from pyspark.sql import functions as F

# Fact: agregar salt a las transacciones
df_fact_salted = (
    df_skew
    .withColumn(
        "salt",
        F.when(
            F.col("customer_key") == 1,
            F.pmod(F.col("id"), F.lit(8))
        ).otherwise(F.lit(0))
    )
)

# Dimensión original: una fila por cliente
df_customer_dimension = (
    df_skew
    .select("customer_key")
    .distinct()
)

# Replicar el cliente caliente 8 veces, una por cada salt
df_hot_customer_dimension = (
    df_customer_dimension
    .filter(F.col("customer_key") == 1)
    .crossJoin(
        spark.range(8).select(F.col("id").alias("salt"))
    )
)

# Los demás clientes conservan una sola fila con salt = 0
df_regular_customer_dimension = (
    df_customer_dimension
    .filter(F.col("customer_key") != 1)
    .withColumn("salt", F.lit(0))
)

# Dimensión final preparada para el JOIN
df_dimension_salted = (
    df_hot_customer_dimension
    .unionByName(df_regular_customer_dimension)
)


In [0]:

print("Transacciones:", df_fact_salted.count())

print("Filas dimensión original:", df_customer_dimension.count())

print("Filas dimensión con salting:", df_dimension_salted.count())

display(
    df_dimension_salted
    .filter(F.col("customer_key") == 1)
    .orderBy("salt")
)


In [0]:
# Saling cond os condiciones

df_join_salted = (
    df_fact_salted.alias("f")
    .join(
        df_dimension_salted.alias("d"),
        (F.col("f.customer_key") == F.col("d.customer_key")) &
        (F.col("f.salt") == F.col("d.salt")),
        "inner"
    )
    .select(
        F.col("f.id"),
        F.col("f.customer_key"),
        F.col("f.salt")
    )
)

print("Registros después del JOIN:", df_join_salted.count())
print("IDs únicos:", df_join_salted.select("id").distinct().count())


In [0]:
df_join_salted.explain("formatted")

In [0]:
df_join_distribution = (
    df_join_salted
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
    .groupBy("partition_id")
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("customer_key").alias("distinct_customers"),
        F.sum(
            F.when(F.col("customer_key") == 1, 1).otherwise(0)
        ).alias("hot_customer_records")
    )
    .orderBy("partition_id")
)

display(df_join_distribution)

In [0]:
from pyspark.sql.functions import broadcast

df_join_shuffle_hash = (
    df_fact_salted.hint("SHUFFLE_HASH").alias("f")
    .join(
        df_dimension_salted.hint("SHUFFLE_HASH").alias("d"),
        (F.col("f.customer_key") == F.col("d.customer_key")) &
        (F.col("f.salt") == F.col("d.salt")),
        "inner"
    )
    .select(
        F.col("f.id"),
        F.col("f.customer_key"),
        F.col("f.salt")
    )
)

print("Registros:", df_join_shuffle_hash.count())

df_join_shuffle_hash.explain("formatted")

In [0]:

df_shuffle_hash_distribution = (
    df_join_shuffle_hash
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("customer_key").alias("distinct_customers"),
        F.sum(
            F.when(F.col("customer_key") == 1, 1).otherwise(0)
        ).alias("hot_customer_records")
    )
    .orderBy("partition_id")
)

display(df_shuffle_hash_distribution)
